In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_tr")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlssmartdatagrupo18")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/transactions.csv"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, DoubleType

transactions_schema = StructType(fields=[
    StructField("transaction_id", StringType(), False),
    StructField("transaction_date", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("store_location", StringType(), True),
    StructField("payment_method", StringType(), True)
])

In [0]:
df_transactions = spark.read \
    .schema(transactions_schema) \
    .option("header", "true") \
    .option("multiLine", True) \
    .csv(ruta)

In [0]:
(
    df_transactions.write.format("delta")
    .mode("overwrite")  # or "append" depending on your pipeline
    .saveAsTable("catalog_tr.bronze.transactions_bronze")
)